In [22]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

In [23]:
path = '../data/raw/ai4i2020_processed.csv'
df = pl.read_csv(path)
print('Dataset loaded')
print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')

df.sample(10)

Dataset loaded
Rows: 9973
Columns: 14


UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
i64,str,i64,f64,f64,i64,f64,i64,i64,i64,i64,i64,i64,i64
3106,"""L50285""",0,299.8,309.1,1612,35.4,205,0,0,0,0,0,0
7901,"""L55080""",0,300.8,312.4,1825,23.8,127,0,0,0,0,0,0
3423,"""L50602""",0,301.5,310.4,1629,34.1,151,0,0,0,0,0,0
925,"""L48104""",0,295.5,306.0,1800,27.6,208,0,0,0,0,0,0
2079,"""M16938""",1,299.4,309.3,1692,28.4,201,0,0,0,0,0,0
7018,"""M21877""",1,300.7,310.6,1474,37.0,33,0,0,0,0,0,0
45,"""M14904""",1,298.8,309.1,1472,47.5,125,0,0,0,0,0,0
932,"""L48111""",0,295.5,305.9,1542,36.2,12,0,0,0,0,0,0
8391,"""M23250""",1,298.8,309.8,1288,61.9,79,0,0,0,0,0,0


In [24]:
print('Separating all column by type:\n')
identifier = [
    'UDI', 'Product ID', 'Type'
]
failure = [
    'TWF', 'HDF', 'PWF', 'OSF', 'RNF'
]
target = [
    'Machine failure'
]
operational = [
    'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]',	'Torque [Nm]', 'Tool wear [min]'
]
print(f'Identifier: {identifier}')
print(f'Failure: {failure}')
print(f'Operational: {operational}')
print(f'Target: {target}')

Separating all column by type:

Identifier: ['UDI', 'Product ID', 'Type']
Failure: ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
Operational: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
Target: ['Machine failure']


In [25]:
from plotly.subplots import make_subplots

fig = make_subplots(
    rows= 3, cols = 2
)

for i, col_name in enumerate(operational):
    row = (i//2) + 1
    col = (i% 2) + 1
    fig.add_trace(
        go.Histogram(
            x = df[col_name]
        ), row= row, col = col
    )
    fig.update_xaxes(
        title_text = col_name,
        row = row,
        col = col
    )

fig.update_layout(
    height = 900,
    title = dict(
        text = 'Univariate Analysis for operational columns - Histogram',
        subtitle = dict(text = 'These charts below show us the frequency for each operational column'),
        x = 0.5
    ),
    title_subtitle_font_size= 14,
    showlegend = False
)

fig.show()

In [26]:
fig = make_subplots(
    rows= 3, cols= 2,
    vertical_spacing= 0.12,
    horizontal_spacing= 0.1
)

for i, col_name in enumerate(operational):
    row = (i // 2) +1
    col = (i % 2) + 1
    for mf_failure, mf_color in [(0, 'red'), (1, 'blue')]:
        df_filtered = df.filter(
            pl.col('Machine failure') == mf_failure
        )
        fig.add_trace(
            go.Box(
                y = df_filtered[col_name],
                marker_color= mf_color,
                name= f'Failure {mf_failure}',
                legendgroup= f'{mf_failure}',
                showlegend= i==1
            ), row= row, col = col
        )
        fig.update_xaxes(
            title_text = col_name,
            row = row, col= row 
        )

fig.update_layout(
    height = 1200,
    title = dict(
        text= 'Univariate Analysis for operational columns - Box Plot',
        subtitle = dict(text = 'These charts below show us the position, dispersion, simmetry and outliers for each operational column'),
        x = 0.5
        ),
    title_subtitle_font_size= 14
    )


fig.show()

In [27]:
print('Starting the Bivariate Analysis\n')
print('Creating a Correlational Matrix between operational columns\nIt shows the linear correlation between columns')
print('+1: Positive perfect correlation\n 0: No correlation\n-1: Negative perfect correlation')

matrix = df.select(pl.col(operational)).corr()
matrix

Starting the Bivariate Analysis

Creating a Correlational Matrix between operational columns
It shows the linear correlation between columns
+1: Positive perfect correlation
 0: No correlation
-1: Negative perfect correlation


Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
f64,f64,f64,f64,f64
1.0,0.876068,0.023332,-0.014553,0.01437
0.876068,1.0,0.0202,-0.015275,0.01397
0.023332,0.0202,1.0,-0.875069,-0.00011
-0.014553,-0.015275,-0.875069,1.0,-0.002552
0.01437,0.01397,-0.00011,-0.002552,1.0


In [28]:
fig = px.imshow(
    matrix,
    labels=dict(color = 'Correlation'),
    text_auto='.3f',
    y= operational,
    x= operational
    )
fig.update_xaxes(side='top', tickangle = 6)
fig.update_layout(
    title=dict(
        text='Correlation Matrix (pearson) - Operational variables',
        subtitle = dict(text = 'Visual form for linear correlation'),
        x = 0.5
    ),
    title_subtitle_font_size= 14,
    width=1450,
    height=800,
    margin = dict(t=110), title_y = 0.98
)

fig.update_layout(
    height = 700,
    title = dict(
        text= 'Univariate Analysis for operational columns - Box Plot',
        subtitle = dict(text = 'These charts below show us the position, dispersion, simmetry and outliers for each operational column'),
        x = 0.5
        ),
    title_subtitle_font_size= 14
    )


fig.show()

In [29]:
print('We can see:')
print('- Positive correlation between Process temperature and Air temperature (0.876);')
print('- Negative correlation between Torque and Rotational speed (0.875).')


We can see:
- Positive correlation between Process temperature and Air temperature (0.876);
- Negative correlation between Torque and Rotational speed (0.875).


In [30]:
import numpy as np

df = df.rename(
        {'Rotational speed [rpm]' : 'Rotational speed [rad/s]'}
    ).with_columns(
    ((pl.col('Rotational speed [rad/s]') *2 * np.pi) / 60).alias('Rotational speed [rad/s]')
    )

print('Adjusting the column Rotational speed from rpm to rad/s')
df.sample(10)

Adjusting the column Rotational speed from rpm to rad/s


UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rad/s],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
i64,str,i64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64
9919,"""L57098""",0,298.4,308.8,141.581109,56.7,24,0,0,0,0,0,0
5304,"""L52483""",0,303.9,313.2,155.089957,44.1,197,0,0,0,0,0,0
9647,"""L56826""",0,299.4,310.4,165.666653,31.7,181,0,0,0,0,0,0
5373,"""H34786""",2,302.9,312.3,158.02211,46.8,175,0,0,0,0,0,0
9168,"""M24027""",1,297.6,308.7,145.769899,41.2,178,0,0,0,0,0,0
9465,"""L56644""",0,298.6,309.4,150.796447,48.2,136,0,0,0,0,0,0
1265,"""M16124""",1,297.8,309.3,234.886411,15.7,36,0,0,0,0,0,0
8209,"""L55388""",0,299.2,310.7,154.671078,44.0,19,0,0,0,0,0,0
7749,"""L54928""",0,300.4,311.7,158.650429,47.4,170,0,0,0,0,0,0


In [31]:

fig = make_subplots(
    rows= 1, cols= 2
)

for fail, mf_color in [(0, 'blue'), (1, 'red')]:
    df_filtered = df.filter(pl.col('HDF') == fail)
    fig.add_trace(
        go.Scatter(
            x = df_filtered['Air temperature [K]'],
            y = df_filtered['Process temperature [K]'],
            mode= 'markers',
            name= f'Failure {fail}',
            marker= dict(color=mf_color),
            legendgroup=f'{fail}'
        ),
        row= 1, col= 1
    )
    df_filtered = df.filter(pl.col('PWF') == fail)
    fig.add_trace(
            go.Scatter(
                x = df_filtered['Torque [Nm]'],
                y = df_filtered['Rotational speed [rad/s]'],
                mode= 'markers',
                name= f'Failure {fail}',
                marker= dict(color=mf_color),
                legendgroup=f'{fail}',
                showlegend= False
            ),
            row= 1, col= 2
        )


fig.update_layout(
    height = 500,
    title = dict(
        text= 'Failure patterns',
        subtitle = dict(text = 'How specific failure appers at correlated columns?'),
        x = 0.5
        ),
    title_subtitle_font_size= 14
    )
fig.update_xaxes(
    title_text = 'Air Temperature [K]',
    row= 1, col= 1
)
fig.update_xaxes(
    title_text = 'Torque [Nm]',
    row= 1, col= 2
)
fig.update_yaxes(
    title_text = 'Process Temperature [K]',
    row= 1, col= 1
)
fig.update_yaxes(
    title_text = 'Rotational speed [rad/s]',
    row= 1, col= 2
)
fig.show()

In [32]:
print('Conclusion:\n')
print('- Air temperature and Process temperature walk together, with a positive correlation of 0.876')
print('- As the chart shows, failure of "Heat Dissipation" appers only when both temperatures goes high\n')
print('- Torque and rotational speed are the oposite, with a negative correlation of 0.875')
print('- As the chart shows, failure of "Power" (Power = Torque x Rotational speed) appers only when one of them are high or low')

print('\n----------------------------------------------\n')
print('For knowledge, the same graphs with all Machine failures below')


Conclusion:

- Air temperature and Process temperature walk together, with a positive correlation of 0.876
- As the chart shows, failure of "Heat Dissipation" appers only when both temperatures goes high

- Torque and rotational speed are the oposite, with a negative correlation of 0.875
- As the chart shows, failure of "Power" (Power = Torque x Rotational speed) appers only when one of them are high or low

----------------------------------------------

For knowledge, the same graphs with all Machine failures below


In [33]:

fig = make_subplots(
    rows= 1, cols= 2
)

for fail, mf_color in [(0, 'blue'), (1, 'red')]:
    df_filtered = df.filter(pl.col('Machine failure') == fail)
    fig.add_trace(
        go.Scatter(
            x = df_filtered['Air temperature [K]'],
            y = df_filtered['Process temperature [K]'],
            mode= 'markers',
            name= f'Failure {fail}',
            marker= dict(color=mf_color),
            legendgroup=f'{fail}'
        ),
        row= 1, col= 1
    )
    df_filtered = df.filter(pl.col('Machine failure') == fail)
    fig.add_trace(
            go.Scatter(
                x = df_filtered['Torque [Nm]'],
                y = df_filtered['Rotational speed [rad/s]'],
                mode= 'markers',
                name= f'Failure {fail}',
                marker= dict(color=mf_color),
                legendgroup=f'{fail}',
                showlegend= False
            ),
            row= 1, col= 2
        )


fig.update_layout(
    height = 500,
    title = dict(
        text= 'Failure patterns',
        subtitle = dict(text = 'How failure appers at correlated columns?'),
        x = 0.5
        ),
    title_subtitle_font_size= 14
    )
fig.update_xaxes(
    title_text = 'Air Temperature [K]',
    row= 1, col= 1
)
fig.update_xaxes(
    title_text = 'Torque [Nm]',
    row= 1, col= 2
)
fig.update_yaxes(
    title_text = 'Process Temperature [K]',
    row= 1, col= 1
)
fig.update_yaxes(
    title_text = 'Rotational speed [rad/s]',
    row= 1, col= 2
)
fig.show()

In [34]:
print('Saving the data with rotational speed in [rad/s] (more accurate for power analysis)\n')

try:
    df.write_csv('../data/processed/ai4i2020_new.csv')
    print('Dataset saved at /data/processed')
except Exception as e:
    print('Fail to save - error: {e}')

Saving the data with rotational speed in [rad/s] (more accurate for power analysis)

Dataset saved at /data/processed
